## make koppen-geiger climate images

In [ ]:
# setup imports
import sys
from pathlib import Path

from tqdm.notebook import tqdm

PROJECT_ROOT = Path().resolve().parents[1]  # pas aan als notebook dieper/dichter zit
sys.path.append(str(PROJECT_ROOT))

from src.paths import *
from src.plotting import plot_koppen_geiger

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from cartopy.feature import BORDERS, COASTLINE, LAKES, OCEAN, RIVERS, ShapelyFeature
from cartopy.io.shapereader import Reader
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
path_to_file = KOPPEN_GEIGER / "1931_1960" / "koppen_geiger_0p00833333.tif"
print(path_to_file)

In [ ]:
# --- Raster ---
# tiff_file = "koppen_map.tiff"
with rasterio.open(path_to_file) as src:
    data = src.read(1)

In [ ]:
class_names = [
    "Af",
    "Am",
    "Aw",
    "BWh",
    "BWk",
    "BSh",
    "BSk",
    "Csa",
    "Csb",
    "Csc",
    "Cwa",
    "Cwb",
    "Cwc",
    "Cfa",
    "Cfb",
    "Cfc",
    "Dsa",
    "Dsb",
    "Dsc",
    "Dsd",
    "Dwa",
    "Dwb",
    "Dwc",
    "Dwd",
    "Dfa",
    "Dfb",
    "Dfc",
    "Dfd",
    "ET",
    "EF",
]


rgb_colors = (
    np.array(
        [
            [0, 0, 255],
            [0, 120, 255],
            [70, 170, 250],
            [255, 0, 0],
            [255, 150, 150],
            [245, 165, 0],
            [255, 220, 100],
            [255, 255, 0],
            [200, 200, 0],
            [150, 150, 0],
            [150, 255, 150],
            [100, 200, 100],
            [50, 150, 50],
            [200, 255, 80],
            [100, 255, 80],
            [50, 200, 0],
            [255, 0, 255],
            [200, 0, 200],
            [150, 50, 150],
            [150, 100, 150],
            [170, 175, 255],
            [90, 120, 220],
            [75, 80, 180],
            [50, 0, 135],
            [0, 255, 255],
            [55, 200, 255],
            [0, 125, 125],
            [0, 70, 95],
            [178, 178, 178],
            [102, 102, 102],
        ]
    )
    / 255
)


cmap = ListedColormap(rgb_colors)
norm = BoundaryNorm(np.arange(0.5, 31.5, 1), cmap.N)

In [ ]:
# --- Shapefiles ---
shapefiles = {
    # "Amu Darya": {"path": SHAPEFILES/"Chatly_GRDC/Chatly_GRDC.shp", "edgecolor": "blue", "linewidth": 2},
    # "Syr Darya": {"path": SHAPEFILES/"Kazalinsk_GRDC/Kazalinsk_GRDC.shp", "edgecolor": "red", "linewidth": 2},
    "Aral Sea Basin": {
        "path": SHAPEFILES / "AralSea_basin/AralSea_basin.shp",
        "edgecolor": "black",
        "linewidth": 2,
    }
}

In [ ]:
# --- Cartopy figure ---
fig = plt.figure(figsize=(12, 8), dpi=300)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([54, 82, 31, 53], crs=ccrs.PlateCarree())

# --- Plot raster ---
ax.imshow(
    data,
    cmap=cmap,
    norm=norm,
    origin="upper",
    extent=[-180, 180, -90, 90],  # full raster extent
    transform=ccrs.PlateCarree(),
)


# --- Add map features ---
ax.add_feature(COASTLINE, linewidth=1, edgecolor="black")
ax.add_feature(BORDERS, linewidth=1, edgecolor="black", linestyle=":")
ax.add_feature(LAKES, facecolor="lightblue", edgecolor="blue", zorder=19)
ax.add_feature(RIVERS, edgecolor="blue", linewidth=1, zorder=15)
ax.add_feature(OCEAN, facecolor="lightblue", edgecolor="blue", zorder=20)


legend_handles = []

# Köppen classes for legend
for i, name in enumerate(class_names):
    patch = Patch(facecolor=rgb_colors[i], edgecolor="k", label=f"{i+1}: {name}")
    legend_handles.append(patch)

# Add shapefiles and legend handles
for label, cfg in shapefiles.items():
    feature = ShapelyFeature(
        Reader(cfg["path"]).geometries(),
        ccrs.PlateCarree(),
        facecolor="none",
        edgecolor=cfg["edgecolor"],
        linewidth=cfg["linewidth"],
    )
    ax.add_feature(feature)
    legend_handles.append(Line2D([0], [0], color=cfg["edgecolor"], linewidth=2, label=label))

# --- Gridlines and labels ---
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color="gray", alpha=0.5, linestyle="--")
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {"size": 10}
gl.ylabel_style = {"size": 10}

# --- Combined legend ---
# plt.legend(handles=legend_handles, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

# --- Title ---
plt.title("Köppen-Geiger Map for Aral Sea Basin", fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# path_to_file = KOPPEN_GEIGER/"1931_1960"/"koppen_geiger_0p00833333.tif"


path_to_file = KOPPEN_GEIGER / "2041_2070" / "ssp370" / "koppen_geiger_0p00833333.tif"
plot_koppen_geiger(path_to_file=path_to_file, show_legend=False, show_plot=True, show_title=False)

In [ ]:
from pathlib import Path

root = KOPPEN_GEIGER

files = list(root.rglob("koppen_geiger_0p00833333.tif"))

for f in files:
    print(f)

In [ ]:
print(len(files))

In [ ]:
save_dir_no_legend = KOPPEN_FIGURES / "no_legend"
save_dir_with_legend = KOPPEN_FIGURES / "with_legend"
save_dir_no_legend.mkdir(parents=True, exist_ok=True)
save_dir_with_legend.mkdir(parents=True, exist_ok=True)

In [ ]:
# n_files = len(files)


# for path in tqdm(sorted(KOPPEN_GEIGER.rglob("koppen_geiger_0p00833333.tif")),
#                  total=n_files, desc="Plotting & saving Köppen maps"):
#     parts = path.parts

#     plot_koppen_geiger(
#         path_to_file=path,
#         savefig=True,
#         save_dir=save_dir_with_legend,
#         show_legend=True,
#         show_plot=False,  # do not display in notebook
#         show_title=False
#     )

In [ ]:
# for path in tqdm(sorted(KOPPEN_GEIGER.rglob("koppen_geiger_0p00833333.tif")),
#                  total=n_files, desc="Plotting & saving Köppen maps"):
#     parts = path.parts

#     plot_koppen_geiger(
#         path_to_file=path,
#         savefig=True,
#         save_dir=save_dir_no_legend,
#         show_legend=False,
#         show_plot=False,  # do not display in notebook
#         show_title=False
#     )

In [ ]:
path_to_file = KOPPEN_GEIGER / "1961_1990" / "koppen_geiger_0p00833333.tif"
plot_koppen_geiger(
    path_to_file=path_to_file, show_legend=True, show_plot=True, show_title=True, savefig=True
)

In [ ]:
from pathlib import Path

from tqdm import tqdm

from src.paths import KOPPEN_FIGURES, KOPPEN_GEIGER, SHAPEFILES
from src.plotting import analyse_koppen_geiger
from src.utils import KOPPEN_DESCRIPTION

# Directories for saving outputs
save_dir = KOPPEN_FIGURES / "analysis"
save_dir.mkdir(parents=True, exist_ok=True)

# Define shapefiles (adjust paths if needed)
shapefiles = {
    "Aral Sea Basin": {
        "path": SHAPEFILES / "AralSea_basin/AralSea_basin.shp",
        "edgecolor": "black",
        "linewidth": 2,
        "linestyle": "-",
    },
    "Chatly": {
        "path": SHAPEFILES / "Chatly_GRDC/Chatly_GRDC.shp",
        "edgecolor": "blue",
        "linewidth": 2,
    },
    "Kazalinsk": {
        "path": SHAPEFILES / "Kazalinsk_GRDC/Kazalinsk_GRDC.shp",
        "edgecolor": "red",
        "linewidth": 2,
    },
}

# Find all raster files
files = sorted(KOPPEN_GEIGER.rglob("koppen_geiger_0p00833333.tif"))

# Loop over each scenario/raster
for path in tqdm(files, total=len(files), desc="Analyzing Köppen-Geiger"):
    fig, ax, df_percent, top_df = analyse_koppen_geiger(
        path_to_file=path,
        shapefiles=shapefiles,
        koppen_description=KOPPEN_DESCRIPTION,
        plot_map=True,  # generate raster map
        plot_hist=True,  # generate histograms
        generate_table=True,  # generate top-N table
        save_dir=save_dir,  # all outputs saved here
    )